In [ ]:
import random
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.metrics import confusion_matrix
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# 2. Device Configuration and Seed Initialization
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

seed = 42

torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)

# 3. Load CIFAR-10 Dataset

# Basic Transforms
transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))]
)

# Download Dataset
train_dataset = datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform
)

test_dataset = datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transform
)

# Create DataLoaders
batch_size = 128

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
# 4. CNN Starter Code
class InitialCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.conv4 = nn.Conv2d(128, 256, 3, padding=1)

        self.fc1 = nn.Linear(256 * 2 * 2, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        # Changing the Activaition function from sigmoid to relu
        #      x = torch.sigmoid(self.conv1(x))
        x = torch.relu(self.conv1(x))
        x = F.max_pool2d(x, 2)

        #     x = torch.sigmoid(self.conv2(x))
        x = torch.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)

        #    x = torch.sigmoid(self.conv3(x))
        x = torch.relu(self.conv3(x))
        x = F.max_pool2d(x, 2)

        #   x = torch.sigmoid(self.conv4(x))
        x = torch.relu(self.conv4(x))
        x = F.max_pool2d(x, 2)

        x = x.view(x.size(0), -1)

        #  x = torch.sigmoid(self.fc1(x))
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)

        return x


# 5. MLP Starter Code
class InitialMLP(nn.Module):

    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(32 * 32 * 3, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)  # flatten

        # Changing the Activation function from sigmoid to relu
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)

        return x


# 7. Training Function
def train(model, loader, optimizer, criterion):

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    gradient_norms = {}

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        # Save gradient norms
        for name, param in model.named_parameters():

            if param.grad is not None:

                grad_norm = param.grad.norm().item()

                if name not in gradient_norms:
                    gradient_norms[name] = []

                gradient_norms[name].append(grad_norm)

        optimizer.step()

        running_loss += loss.item()

        _, predicted = outputs.max(1)

        total += labels.size(0)

        correct += predicted.eq(labels).sum().item()

    accuracy = 100 * correct / total

    avg_loss = running_loss / len(loader)

    return avg_loss, accuracy, gradient_norms


# 8. Testing Function
def test(model, loader, criterion):

    model.eval()

    running_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item()

            _, predicted = outputs.max(1)

            total += labels.size(0)

            correct += predicted.eq(labels).sum().item()

    accuracy = 100 * correct / total

    avg_loss = running_loss / len(loader)

    return avg_loss, accuracy

In [ ]:
# 9. Main Training Loop (CNN)
num_epochs = 15
criterion = nn.CrossEntropyLoss()

# 6. Initialize Network (CNN)
cnn_model = InitialCNN().to(device)
cnn_optimizer = optim.Adam(cnn_model.parameters(), lr=0.001)

train_losses = []
test_losses = []
train_accs = []
test_accs = []
all_epochs_grad_norms = []  # To store gradient norms for all epochs

print("Initiating the training of the CNN model")
for epoch in range(num_epochs):

    train_loss, train_acc, grad_norms = train(
        cnn_model, train_loader, cnn_optimizer, criterion
    )

    all_epochs_grad_norms.append(
        grad_norms
    )  # Store grad_norms for the current epoch

    test_loss, test_acc = test(cnn_model, test_loader, criterion)

    train_losses.append(train_loss)
    test_losses.append(test_loss)

    train_accs.append(train_acc)
    test_accs.append(test_acc)

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Accuracy: {train_acc:.2f}%")
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test Accuracy: {test_acc:.2f}%")
    print("-" * 40)


# 10. Train the MLP & Epoches output for trained MLP
mlp_model = InitialMLP().to(device)
mlp_optimizer = optim.Adam(mlp_model.parameters(), lr=0.001)

mlp_train_losses = []
mlp_test_losses = []
mlp_train_accs = []
mlp_test_accs = []

print("Initiating the training the MLP model")
for epoch in range(num_epochs):
    train_loss, train_acc, _ = train(
        mlp_model, train_loader, mlp_optimizer, criterion
    )

    test_loss, test_acc = test(mlp_model, test_loader, criterion)

    mlp_train_losses.append(train_loss)
    mlp_test_losses.append(test_loss)
    mlp_train_accs.append(train_acc)
    mlp_test_accs.append(test_acc)

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"MLP Train Loss: {train_loss:.4f} | MLP Train Accuracy: {train_acc:.2f}%")
    print(f"MLP Test Loss: {test_loss:.4f} | MLP Test Accuracy: {test_acc:.2f}%")
    print("-" * 50)

In [ ]:
# CNN and MLP weights vis.

# Extract the weights of the first layer in the MLP
mlp_weights = mlp_model.fc1.weight.detach().cpu()

# Set up the canvas to display 10 neurons (a 2x5 grid)
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle(
    "MLP First Layer Weights Visualization (Reshaped to 32x32x3)", fontsize=14
)

for i, ax in enumerate(axes.flat):
    # Extract the weight vector for the i-th neuron (length 3072)
    weight_vector = mlp_weights[i]

    # Reshape the vector back to a color image structure (C, H, W)
    img = weight_vector.view(3, 32, 32)

    # Normalize values to the 0-1 range for proper display
    img = (img - img.min()) / (img.max() - img.min())

    # Permute dimension order from (C, H, W) to (H, W, C) for matplotlib
    img = img.permute(1, 2, 0).numpy()

    ax.imshow(img)
    ax.set_title(f"Neuron {i+1}")
    ax.axis("off")

plt.show()

# Extract the weights of the first layer in the CNN
cnn_weights = cnn_model.conv1.weight.detach().cpu()

# Set up the canvas (a 4x8 grid for the 32 filters)
fig, axes = plt.subplots(4, 8, figsize=(12, 6))
fig.suptitle("CNN First Layer Filter Weights Visualization", fontsize=14)

for i, ax in enumerate(axes.flat):
    if i < cnn_weights.size(0):
        # Extract the i-th filter
        img = cnn_weights[i]

        # Normalize filter weight values to the 0-1 range for proper color display
        img = (img - img.min()) / (img.max() - img.min())

        # Permute dimension order from (C, H, W) to (H, W, C) for matplotlib
        img = img.permute(1, 2, 0).numpy()

        ax.imshow(img)
    ax.axis("off")

plt.show()


# Learning curves

# Set up a canvas with 1 row and 2 columns for side-by-side comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
epochs = range(1, 16)  # 15 epochs

# Graph 7: Loss Comparison (CNN vs MLP)

# CNN curves
ax1.plot(
    epochs,
    train_losses,
    label="CNN Train Loss",
    color="#1f77b4",
    linestyle="-",
    linewidth=2,
)
ax1.plot(
    epochs,
    test_losses,
    label="CNN Test Loss",
    color="#ff7f0e",
    linestyle="-",
    linewidth=2,
)

# MLP curves
ax1.plot(
    epochs,
    mlp_train_losses,
    label="MLP Train Loss",
    color="#1f77b4",
    linestyle="--",
    linewidth=2,
)
ax1.plot(
    epochs,
    mlp_test_losses,
    label="MLP Test Loss",
    color="#ff7f0e",
    linestyle="--",
    linewidth=2,
)

ax1.set_title(
    "Graph 7: Train & Test Loss Comparison (CNN vs MLP)",
    fontsize=12,
    fontweight="bold",
)
ax1.set_xlabel("Epoch", fontsize=11)
ax1.set_ylabel("Loss", fontsize=11)
ax1.set_xticks(epochs)
ax1.grid(True, linestyle=":", alpha=0.6)
ax1.legend(fontsize=10, loc="upper right")

# Graph 8: Accuracy Comparison (CNN vs MLP)

# CNN curves
ax2.plot(
    epochs,
    train_accs,
    label="CNN Train Accuracy",
    color="#2ca02c",
    linestyle="-",
    linewidth=2,
)
ax2.plot(
    epochs,
    test_accs,
    label="CNN Test Accuracy",
    color="#d62728",
    linestyle="-",
    linewidth=2,
)

# MLP curves
ax2.plot(
    epochs,
    mlp_train_accs,
    label="MLP Train Accuracy",
    color="#2ca02c",
    linestyle="--",
    linewidth=2,
)
ax2.plot(
    epochs,
    mlp_test_accs,
    label="MLP Test Accuracy",
    color="#d62728",
    linestyle="--",
    linewidth=2,
)

ax2.set_title(
    "Graph 8: Train & Test Accuracy Comparison (CNN vs MLP)",
    fontsize=12,
    fontweight="bold",
)
ax2.set_xlabel("Epoch", fontsize=11)
ax2.set_ylabel("Accuracy (%)", fontsize=11)
ax2.set_xticks(epochs)
ax2.grid(True, linestyle=":", alpha=0.6)
ax2.legend(fontsize=10, loc="lower right")

# Adjust layout and display the visualization
plt.tight_layout()
plt.show()


# Confusion Matrix for evaluation (CNN VS MLP)

# Confusion matrix MLP


def plot_mlp_confusion_matrix(mlp_model, loader, device, title="MLP Confusion Matrix"):
    """
    Evaluates the trained MLP model on the test dataset and plots a
    normalized confusion matrix to analyze classification errors.
    """
    # Set the model to evaluation mode (disables dropout/batchnorm updates)
    mlp_model.eval()

    all_preds = []
    all_labels = []

    # Disable gradient computation for faster inference and memory efficiency
    with torch.no_grad():
        for images, labels in loader:
            # Move data to the active hardware accelerator (GPU or CPU)
            images = images.to(device)

            # Forward pass: compute predicted outputs by passing images to the model
            outputs = mlp_model(images)

            # Get the index of the highest log-probability (predicted class)
            _, preds = outputs.max(1)

            # Collect predictions and true labels back to CPU as numpy arrays
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    # Official target class names ordered by CIFAR-10 indexing
    class_names = [
        "airplane",
        "automobile",
        "bird",
        "cat",
        "deer",
        "dog",
        "frog",
        "horse",
        "ship",
        "truck",
    ]

    # Generate the raw confusion matrix mapping actual vs. predicted labels
    raw_cm = confusion_matrix(all_labels, all_preds)

    # Normalize the matrix rows to display percentages instead of raw sample counts
    normalized_cm = raw_cm.astype("float") / raw_cm.sum(axis=1)[:, np.newaxis] * 100

    # Initialize a clean canvas for heat map plotting
    plt.figure(figsize=(10, 8))

    # Render the confusion matrix using seaborn's annotated heatmap
    sns.heatmap(
        normalized_cm,
        annot=True,
        fmt=".1f",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
    )

    # Formatting titles and axes labels for the academic report
    plt.title(title, fontsize=14, fontweight="bold")
    plt.ylabel("True Label", fontsize=12)
    plt.xlabel("Predicted Label", fontsize=12)

    # Optimize layout geometry and present the final figure
    plt.tight_layout()
    plt.show()


# Execute the visualization using your trained MLP model
# We label this as Graph 9 to be presented right before the CNN matrix
plot_mlp_confusion_matrix(
    mlp_model,  # Your trained MLP instance from the current notebook
    test_loader,  # The evaluation DataLoader pulled from the .py file
    device,  # Target device (CUDA or CPU)
    title="Confusion Matrix 4: MLP Confusion Matrix",
)


# Confusion matrix CNN


def plot_cnn_confusion_matrix(cnn_model, loader, device, title="CNN Confusion Matrix"):
    """
    Evaluates the trained CNN model on the test dataset and plots a
    normalized confusion matrix to analyze classification errors.
    """
    # Set the model to evaluation mode (disables dropout/batchnorm updates)
    cnn_model.eval()

    all_preds = []
    all_labels = []

    # Disable gradient computation for faster inference and memory efficiency
    with torch.no_grad():
        for images, labels in loader:
            # Move data to the active hardware accelerator (GPU or CPU)
            images = images.to(device)

            # Forward pass: compute predicted outputs by passing images to the model
            outputs = cnn_model(images)

            # Get the index of the highest log-probability (predicted class)
            _, preds = outputs.max(1)

            # Collect predictions and true labels back to CPU as numpy arrays
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    # Official target class names ordered by CIFAR-10 indexing
    class_names = [
        "airplane",
        "automobile",
        "bird",
        "cat",
        "deer",
        "dog",
        "frog",
        "horse",
        "ship",
        "truck",
    ]

    # Generate the raw confusion matrix mapping actual vs. predicted labels
    raw_cm = confusion_matrix(all_labels, all_preds)

    # Normalize the matrix rows to display percentages instead of raw sample counts
    normalized_cm = raw_cm.astype("float") / raw_cm.sum(axis=1)[:, np.newaxis] * 100

    # Initialize a clean canvas for heat map plotting
    plt.figure(figsize=(10, 8))

    # Render the confusion matrix using seaborn's annotated heatmap
    sns.heatmap(
        normalized_cm,
        annot=True,
        fmt=".1f",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
    )

    # Formatting titles and axes labels for the academic report
    plt.title(title, fontsize=14, fontweight="bold")
    plt.ylabel("True Label", fontsize=12)
    plt.xlabel("Predicted Label", fontsize=12)

    # Optimize layout geometry and present the final figure
    plt.tight_layout()
    plt.show()


# Execute the visualization using the trained CNN model pulled from python script
# We label this as Graph 10 assuming  MLP was Graph 9
plot_cnn_confusion_matrix(
    cnn_model,  # The trained CNN instance from the current notebook
    test_loader,  # The evaluation DataLoader
    device,  # Target device (CUDA or CPU)
    title="Confusion Matrix 3: CNN Confusion Matrix",
)